# OpenPlaque — Image-Driven Tracking with Cache Controls

Complete stepwise Colab. Every persistent cache has a Boolean reuse variable near the top. All default to `True`.

**Rule:** `True` reuses a valid cache if present; otherwise it recomputes and caches. `False` forces one recomputation in this run and updates the cache. Results are also retained in memory, so downstream steps do not recompute the same component again.

Canonical TPV is unchanged. CPR plaque views and source-volume PCAT are not spatially co-registered. Research use only.


## Step 1 — Mount Drive


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache / reuse variables


In [ ]:
REUSE_SERIES_SELECTION = True
REUSE_PLAQUE_MASKS = True
REUSE_TRACKING = True
REUSE_ROADMAPS = True
REUSE_PCAT_FIGURES = True
REUSE_DASHBOARD = True
REUSE_REPORT_PACKAGE = True


## Step 3 — Install this branch


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch image-driven-tracking-cache-controls-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt
import sys
sys.path.insert(0, '/content/OpenPlaque/src')
import pandas as pd
from IPython.display import display, Image
from openplaque.cache_controlled_tracking import CacheControlledTrackingWorkflow
print('Ready.')


## Step 4 — Initialize and inspect cache plan


In [ ]:
REUSE = {
    'series_selection': REUSE_SERIES_SELECTION,
    'plaque_masks': REUSE_PLAQUE_MASKS,
    'tracking': REUSE_TRACKING,
    'roadmaps': REUSE_ROADMAPS,
    'pcat_figures': REUSE_PCAT_FIGURES,
    'dashboard': REUSE_DASHBOARD,
    'report_package': REUSE_REPORT_PACKAGE,
}
wf = CacheControlledTrackingWorkflow('/content/drive/MyDrive/OpenPlaque', REUSE)
display(wf.cache_status())


## Step 5 — Prepare DICOM and series selection


In [ ]:
series_map = wf.prepare_inputs()
print('Series:', series_map)


## Step 6 — Plaque masks


In [ ]:
cache_qc = wf.load_cached_plaque()
display(cache_qc)


## Step 7 — Image-driven coronary tracking


In [ ]:
tracking_qc = wf.track_coronaries()
display(tracking_qc)


## Step 8 — Review top candidates


In [ ]:
candidate_png = wf.plot_candidates()
print(candidate_png)


## Step 9 — Straightened plaque roadmaps


In [ ]:
roadmap_png = wf.plot_straightened_roadmaps()
print(roadmap_png)
if wf.along_df is not None: display(wf.along_df.head(30))


## Step 10 — RCA PCAT figures


In [ ]:
pcat_files = wf.reuse_pcat_outputs()
print(pcat_files)
display(Image(filename=str(pcat_files['cross_sections'])))
display(Image(filename=str(pcat_files['ribbon'])))


## Step 11 — Dashboard


In [ ]:
dashboard = wf.plot_dashboard()
print(dashboard)
display(Image(filename=str(dashboard)))


## Step 12 — Package report


In [ ]:
zip_path = wf.package_report()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_IMAGE_DRIVEN_TRACKING_REPORT_BACK.zip')
print('\nCache provenance:')
display(pd.read_csv(wf.out / 'cache_provenance.csv'))
